# 03 — Allocation diagnostic

Run all five diagnostic sections (combined-book exposure, correlation,
redundancy, risk contribution, benchmark comparison) on the resolved book
and export `reports/allocation_diagnostic.html`.


In [1]:
from datetime import date
from pathlib import Path
import warnings

import pandas as pd

from hailmary.allocation.book_config import ROLES
from hailmary.allocation.diagnostic import (
    benchmark_comparison, combined_exposure, combined_exposure_figure,
    correlation_figure, correlation_matrix, redundancy_pairs,
    render_html_report, risk_contribution,
)
from hailmary.allocation.portfolios import Role, from_parsed
from hailmary.allocation.statements import parse_statement
from hailmary.data.providers import YahooFinanceProvider

STATEMENT_PATH = Path('../../data/statements/2026-04 StashAway Monthly Statement.pdf')
REPORT_PATH = Path('../../reports/allocation_diagnostic.html')
START = date(2022, 1, 1)
END = date.today()
REDUNDANCY_THRESHOLD = 0.85

## Parse + tag + fetch returns

In [2]:
parsed = parse_statement(STATEMENT_PATH, use_cache=False)
portfolios = [from_parsed(p, roles=ROLES[p.name]) for p in parsed if p.name in ROLES]
holding = [p for p in portfolios if Role.HOLDING in p.roles]
tickers = sorted({
    h.metadata.ticker for p in holding for h in p.holdings
    if not h.metadata.ticker.startswith('CASH_')
})
provider = YahooFinanceProvider()
returns = provider.get_returns(tickers, START, END)
print(f'Resolved {len(portfolios)} portfolios; fetched {returns.shape[1]} ticker series')

2026-05-11 01:07:53.186 | DEBUG    | hailmary.data.cache:get:38 - Cache hit key=2b98582336e0


Resolved 15 portfolios; fetched 48 ticker series


## Combined-book exposure

In [3]:
with warnings.catch_warnings():
    warnings.simplefilter('ignore', UserWarning)
    exposure = combined_exposure(portfolios)
for dim, df in exposure.items():
    print(f'\n--- {dim.replace("_", " ").title()} ---')
    display(df)
combined_exposure_figure(exposure)


--- Asset Class ---


,bucket,value,weight
0,Equity,493043.15,0.754534
1,Crypto,107445.47,0.164430
2,Commodity,26155.95,0.040028
3,Bond,18893.00,0.028913
4,Cash,7903.05,0.012095



--- Region ---


,bucket,value,weight
0,US,266730.73,0.408194
1,Global,216117.91,0.330738
2,Developed ex-US,68464.96,0.104776
3,Japan,31107.54,0.047606
4,Singapore,24846.13,0.038024
5,Emerging Markets,20305.17,0.031074
6,India,11899.27,0.018210
7,Asia ex-Japan,5590.93,0.008556
8,Emerging Markets ex-China,3256.74,0.004984
9,Eurozone,2075.77,0.003177



--- Sector ---


,bucket,value,weight
0,Broad Market,306149.57,0.468519
1,Bitcoin (proxy: BTC-USD spot),54778.04,0.083830
2,Ethereum (proxy: ETH-USD spot),52667.43,0.080600
3,Technology,31193.15,0.047737
4,Gold,26155.95,0.040028
5,Dividend,23858.50,0.036512
6,Hedged,21353.80,0.032679
7,Consumer Staples,19655.78,0.030080
8,Energy,16764.32,0.025655
9,Industrials,16367.65,0.025048


## Correlation matrix

In [4]:
with warnings.catch_warnings():
    warnings.simplefilter('ignore', UserWarning)
    corr = correlation_matrix(portfolios, returns=returns)
display(corr.round(3))
correlation_figure(corr)

,BlackRock,Energy,General Investing,Singapore Investing,Utilities,Income Investing,High Dividend Yield,Ex-US Large-cap,SG ETF,Nasdaq Covered Call,Guitsa,Simple SGD,General SRS,Crypto
BlackRock,1.000,0.200,0.696,0.385,0.238,0.423,0.514,0.674,0.347,0.545,NaN,NaN,0.707,0.300
Energy,0.200,1.000,0.403,-0.139,0.279,0.147,0.598,0.376,-0.013,0.351,NaN,NaN,0.405,0.191
General Investing,0.696,0.403,1.000,-0.005,0.415,0.548,0.812,0.881,0.116,0.829,NaN,NaN,0.999,0.737
Singapore Investing,0.385,-0.139,-0.005,1.000,-0.008,-0.068,-0.074,0.020,0.793,-0.082,NaN,NaN,-0.000,-0.083
Utilities,0.238,0.279,0.415,-0.008,1.000,0.450,0.620,0.410,0.005,0.352,NaN,NaN,0.422,0.164
Income Investing,0.423,0.147,0.548,-0.068,0.450,1.000,0.508,0.591,0.028,0.497,NaN,NaN,0.556,0.275
High Dividend Yield,0.514,0.598,0.812,-0.074,0.620,0.508,1.000,0.767,0.050,0.734,NaN,NaN,0.821,0.400
Ex-US Large-cap,0.674,0.376,0.881,0.020,0.410,0.591,0.767,1.000,0.161,0.741,NaN,NaN,0.893,0.450
SG ETF,0.347,-0.013,0.116,0.793,0.005,0.028,0.050,0.161,1.000,0.028,NaN,NaN,0.121,-0.011
Nasdaq Covered Call,0.545,0.351,0.829,-0.082,0.352,0.497,0.734,0.741,0.028,1.000,NaN,NaN,0.837,0.449


## Redundancy (threshold default 0.85)

In [5]:
pairs = redundancy_pairs(corr, threshold=REDUNDANCY_THRESHOLD, portfolios=portfolios)
if pairs:
    pd.DataFrame(pairs, columns=['a', 'b', 'rho', 'candidate'])
else:
    print(f'No portfolio pairs above ρ = {REDUNDANCY_THRESHOLD}.')
    print('If your customs are uncorrelated by design, this is expected — drop the threshold to 0.7 to surface near-redundancy.')

## Risk contribution

In [6]:
with warnings.catch_warnings():
    warnings.simplefilter('ignore', UserWarning)
    risk = risk_contribution(portfolios, returns=returns)
print('--- By portfolio ---')
display(risk['by_portfolio'].round(4))
print('--- By holding (top 15) ---')
display(risk['by_holding'].head(15).round(4))

--- By portfolio ---


,name,weight,contribution,pct_total
0,General Investing,0.5686,0.0832,0.5431
1,Crypto,0.0839,0.0340,0.2219
2,General SRS,0.1550,0.0221,0.1440
3,High Dividend Yield,0.0365,0.0037,0.0242
4,BlackRock,0.0600,0.0037,0.0240
5,Nasdaq Covered Call,0.0121,0.0016,0.0104
6,Ex-US Large-cap,0.0120,0.0016,0.0102
7,Energy,0.0126,0.0010,0.0068
8,Utilities,0.0060,0.0003,0.0021
9,Income Investing,0.0077,0.0001,0.0008


--- By holding (top 15) ---


,name,weight,contribution,pct_total
0,Crypto · ETH-USD,0.0406,0.0199,0.1298
1,General Investing · ETH-USD,0.0325,0.0159,0.1038
2,Crypto · BTC-USD,0.0424,0.0141,0.0921
3,General Investing · BTC-USD,0.0331,0.0110,0.0720
4,General Investing · VEU,0.0722,0.0094,0.0612
5,General Investing · IVV,0.0672,0.0091,0.0593
6,General Investing · XLK,0.0374,0.0073,0.0479
7,General Investing · ISAC.L,0.0920,0.0067,0.0435
8,High Dividend Yield · VYM,0.0365,0.0037,0.0242
9,General SRS · ETH-USD,0.0075,0.0037,0.0240


## Benchmark comparison

In [7]:
with warnings.catch_warnings():
    warnings.simplefilter('ignore', UserWarning)
    bench = benchmark_comparison(portfolios, returns=returns)
bench.round(3)

,sharpe,max_dd,annualised_vol,n_days,sharpe_delta_vs_BlackRock,max_dd_delta_vs_BlackRock,vol_delta_vs_BlackRock,sharpe_delta_vs_General Investing,max_dd_delta_vs_General Investing,vol_delta_vs_General Investing,sharpe_delta_vs_Singapore Investing,max_dd_delta_vs_Singapore Investing,vol_delta_vs_Singapore Investing,sharpe_delta_vs_Income Investing,max_dd_delta_vs_Income Investing,vol_delta_vs_Income Investing
BlackRock,0.981,-0.111,0.120,827.0,0.000,0.000,0.000,0.230,0.067,-0.042,-1.874,-0.086,0.083,0.224,-0.013,0.050
Energy,0.977,-0.251,0.257,851.0,-0.003,-0.141,0.137,0.227,-0.074,0.096,-1.878,-0.226,0.220,0.220,-0.154,0.188
General Investing,0.750,-0.178,0.162,829.0,-0.230,-0.067,0.042,0.000,0.000,0.000,-2.105,-0.153,0.125,-0.007,-0.080,0.092
Singapore Investing,2.855,-0.025,0.037,439.0,1.874,0.086,-0.083,2.105,0.153,-0.125,0.000,0.000,0.000,2.098,0.073,-0.033
Utilities,0.839,-0.230,0.179,851.0,-0.142,-0.120,0.059,0.088,-0.053,0.017,-2.017,-0.205,0.142,0.082,-0.133,0.109
Income Investing,0.757,-0.098,0.070,851.0,-0.224,0.013,-0.050,0.007,0.080,-0.092,-2.098,-0.073,0.033,0.000,0.000,0.000
High Dividend Yield,0.646,-0.150,0.147,851.0,-0.335,-0.040,0.027,-0.105,0.027,-0.014,-2.209,-0.125,0.111,-0.111,-0.053,0.078
Ex-US Large-cap,0.761,-0.176,0.170,851.0,-0.220,-0.066,0.050,0.010,0.001,0.009,-2.094,-0.151,0.133,0.004,-0.079,0.101
SG ETF,1.804,-0.095,0.116,848.0,0.823,0.015,-0.004,1.053,0.082,-0.045,-1.052,-0.070,0.079,1.047,0.002,0.047
Nasdaq Covered Call,0.167,-0.209,0.176,784.0,-0.813,-0.099,0.056,-0.583,-0.032,0.014,-2.688,-0.184,0.139,-0.590,-0.112,0.106


## Export HTML report

In [8]:
with warnings.catch_warnings():
    warnings.simplefilter('ignore', UserWarning)
    out = render_html_report(
        portfolios,
        REPORT_PATH,
        returns=returns,
        redundancy_threshold=REDUNDANCY_THRESHOLD,
        title='Stashaway book — allocation diagnostic',
    )
print(f'Wrote {out.resolve()}')

Wrote C:\Users\Dalva\src\project-hail-mary\reports\allocation_diagnostic.html
